In [1]:
from pathlib import Path
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient

TRACKING_URI = "http://127.0.0.1:5000"
DATA_DIR = Path("/home/ab/Documents/repos/sagemaker-mlops-with-terraform/cost comparison data")

# Input run lists (already curated in your existing CSVs)
RUNS_10K_PATH = DATA_DIR / "runs_local.csv"
RUNS_FULL_PATH = DATA_DIR / "runs_local_full_dataset.csv"

# Output files with per-epoch metrics pulled directly from MLflow
OUT_10K_EPOCHS = DATA_DIR / "runs_local_epochs.csv"
OUT_FULL_EPOCHS = DATA_DIR / "runs_local_full_dataset_epochs.csv"
OUT_ALL_EPOCHS = DATA_DIR / "runs_local_all_epochs.csv"

mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient(tracking_uri=TRACKING_URI)


def map_model_name(run_name: str) -> str:
    run_name = (run_name or "").lower()
    if "f2llm" in run_name:
        return "F2LLM-v2-160M"
    if "gist" in run_name:
        return "GIST-small"
    if "minilm" in run_name:
        return "MiniLM-L6-v2"
    if "static" in run_name:
        return "static-retrieval-mrl"
    return "Unknown"


def collect_epoch_metrics_for_run(run_id: str, dataset_label: str):
    run = client.get_run(run_id)
    run_name = run.data.tags.get("mlflow.runName", run_id)

    metric_keys = [
        "epoch",
        "loss",
        "eval_loss",
        "eval_spearman_cosine",
        "eval_pearson_cosine",
        "learning_rate",
        "grad_norm",
    ]

    # Build row per step and merge available metric points into that row
    by_step = {}
    for key in metric_keys:
        history = client.get_metric_history(run_id, key)
        for point in history:
            row = by_step.setdefault(point.step, {})
            row[key] = point.value
            row[f"{key}_timestamp_ms"] = point.timestamp

    rows = []
    for step in sorted(by_step.keys()):
        row = by_step[step]
        row.update(
            {
                "run_id": run_id,
                "run_name": run_name,
                "model": map_model_name(run_name),
                "dataset": dataset_label,
                "step": step,
            }
        )
        rows.append(row)

    return rows


# Load run IDs from current comparison CSVs
runs_10k_df = pd.read_csv(RUNS_10K_PATH)
runs_full_df = pd.read_csv(RUNS_FULL_PATH)

run_ids_10k = runs_10k_df["Run ID"].dropna().astype(str).tolist()
run_ids_full = runs_full_df["Run ID"].dropna().astype(str).tolist()

print(f"Found {len(run_ids_10k)} run IDs in {RUNS_10K_PATH.name}")
print(f"Found {len(run_ids_full)} run IDs in {RUNS_FULL_PATH.name}")

# Pull full metric histories for each run ID
rows_10k = []
for run_id in run_ids_10k:
    rows_10k.extend(collect_epoch_metrics_for_run(run_id, dataset_label="10k limit"))

rows_full = []
for run_id in run_ids_full:
    rows_full.extend(collect_epoch_metrics_for_run(run_id, dataset_label="full dataset"))

# Save outputs
epochs_10k_df = pd.DataFrame(rows_10k).sort_values(["run_name", "step"]).reset_index(drop=True)
epochs_full_df = pd.DataFrame(rows_full).sort_values(["run_name", "step"]).reset_index(drop=True)
epochs_all_df = pd.concat([epochs_10k_df, epochs_full_df], ignore_index=True)

epochs_10k_df.to_csv(OUT_10K_EPOCHS, index=False)
epochs_full_df.to_csv(OUT_FULL_EPOCHS, index=False)
epochs_all_df.to_csv(OUT_ALL_EPOCHS, index=False)

print("\nSaved files:")
print(f"- {OUT_10K_EPOCHS}")
print(f"- {OUT_FULL_EPOCHS}")
print(f"- {OUT_ALL_EPOCHS}")

print("\nRows saved:")
print(f"- {OUT_10K_EPOCHS.name}: {len(epochs_10k_df)}")
print(f"- {OUT_FULL_EPOCHS.name}: {len(epochs_full_df)}")
print(f"- {OUT_ALL_EPOCHS.name}: {len(epochs_all_df)}")

print("\nEpoch coverage by run (min/max epoch):")
for run_name, g in epochs_all_df.groupby("run_name"):
    if "epoch" in g.columns and g["epoch"].notna().any():
        print(f"- {run_name}: {g['epoch'].min():.3f} -> {g['epoch'].max():.3f}")
    else:
        print(f"- {run_name}: no epoch metric found")

/home/ab/Documents/repos/sagemaker-mlops-with-terraform/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/04/01 01:17:08 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range
2026/04/01 01:17:08 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range
2026/04/01 01:17:08 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range


Found 4 run IDs in runs_local.csv
Found 2 run IDs in runs_local_full_dataset.csv


2026/04/01 01:17:08 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range
2026/04/01 01:17:08 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range
2026/04/01 01:17:08 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range
2026/04/01 01:17:08 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range
2026/04/01 01:17:08 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range
2026/04/01 01:17:08 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range
2026/04/01 01:17:08 WARNING mlflow.tracking.request_header.regis


Saved files:
- /home/ab/Documents/repos/sagemaker-mlops-with-terraform/cost comparison data/runs_local_epochs.csv
- /home/ab/Documents/repos/sagemaker-mlops-with-terraform/cost comparison data/runs_local_full_dataset_epochs.csv
- /home/ab/Documents/repos/sagemaker-mlops-with-terraform/cost comparison data/runs_local_all_epochs.csv

Rows saved:
- runs_local_epochs.csv: 1452
- runs_local_full_dataset_epochs.csv: 26778
- runs_local_all_epochs.csv: 28230

Epoch coverage by run (min/max epoch):
- f2llm-v2-160m-embedding-2026-03-27-06:14:15: 0.000 -> 5.000
- f2llm-v2-160m-embedding-2026-03-29-21:35:14: 0.000 -> 5.000
- gist-small-embedding-2026-03-26-19:47:50: 0.000 -> 5.000
- minilm-embedding-2026-03-25-20:12:41: 0.000 -> 5.000
- minilm-embedding-2026-03-29-14:00:43: 0.000 -> 5.000
- static-embedding-2026-03-25-20:31:32: 0.000 -> 3.000
